Assignment 7.1: Data Analysis and Preparation

Step 1: Import the Required Libraries and Load the Dataset

In this step, the pandas library is imported to support data manipulation and analysis. The immigration dataset is downloaded from an Excel spreadsheet and loaded into a DataFrame for examination.

In [1]:
import pandas as pd

# Specify the source of the Canadian immigration dataset
file_url = 'https://docs.google.com/spreadsheets/d/'\
           '1OGs8KoHGEZDxyFdaxAKmZbeO1BiSWVQ6/'\
           'export?format=xlsx'

# Read the worksheet while skipping its introductory rows
df = pd.read_excel(
    file_url,
    sheet_name='Canada by Citizenship',
    skiprows=20
)

Step 2: Explore the Dataset Structure

The dataset is examined to understand its dimensions, column names, and sample records. Reviewing the beginning and end of the table provides an initial understanding of how the information is organized.

In [2]:
# Display the number of rows and columns
print("Dataset dimensions:", df.shape)

# Review the first and last five records
print(df.head())
print(df.tail())

# List all available column names
print(list(df.columns))

Dataset dimensions: (197, 43)
         Type    Coverage          OdName  AREA AreaName   REG  \
0  Immigrants  Foreigners     Afghanistan   935     Asia  5501   
1  Immigrants  Foreigners         Albania   908   Europe   925   
2  Immigrants  Foreigners         Algeria   903   Africa   912   
3  Immigrants  Foreigners  American Samoa   909  Oceania   957   
4  Immigrants  Foreigners         Andorra   908   Europe   925   

           RegName  DEV             DevName  1980  ...  2004  2005  2006  \
0    Southern Asia  902  Developing regions    16  ...  2978  3436  3009   
1  Southern Europe  901   Developed regions     1  ...  1450  1223   856   
2  Northern Africa  902  Developing regions    80  ...  3616  3626  4807   
3        Polynesia  902  Developing regions     0  ...     0     0     1   
4  Southern Europe  901   Developed regions     0  ...     0     0     1   

   2007  2008  2009  2010  2011  2012  2013  
0  2652  2111  1746  1758  2203  2635  2004  
1   702   560   716   56

Step 3: Inspect Data Types and Record Categories

The data types and missing values are reviewed to assess the dataset's structure and completeness. Records representing aggregate information are also identified to distinguish them from individual-country records.

In [3]:
# Summarize the data types present in the dataset
print(df.dtypes.value_counts())

# Count all missing entries across the table
print("\nTotal missing entries:", df.isnull().sum().sum())

# Identify records that do not represent individual countries
print("\nRecords outside individual-country coverage:")
print(
    df[df['Coverage'] != 'Foreigners']
    [['OdName', 'Coverage']]
)

int64    37
str       6
Name: count, dtype: int64

Total missing entries: 0

Records outside individual-country coverage:
     OdName  Coverage
32   Canada  Citizens
196   Total      Both


Step 4: Clean and Organize the Dataset

The overall aggregate record is preserved separately for validation and excluded from the country-level dataset. Unnecessary columns are removed, descriptive names are assigned to the remaining fields, and country names are used as the index.

In [4]:
# Save the original aggregate record for later comparison
un_total_row = df[df['OdName'] == 'Total']

# Remove the aggregate row from the country-level records
df = df[df['OdName'] != 'Total'].copy()

# Delete fields that are not needed for the analysis
df = df.drop(
    columns=['Type', 'Coverage', 'AREA', 'REG', 'DEV']
)

# Replace abbreviated field names with clearer descriptions
df = df.rename(columns={
    'OdName': 'Country',
    'AreaName': 'Continent',
    'RegName': 'Region'
})

# Convert column labels to strings for consistent year selection
df.columns = df.columns.map(str)

# Use country names to identify individual records
df = df.set_index('Country')

# Check the updated structure
print("Dimensions after preparation:", df.shape)
print(list(df.columns[:5]))

Dimensions after preparation: (196, 37)
['Continent', 'Region', 'DevName', '1980', '1981']


Step 5: Calculate Total Immigration by Country

The annual immigration figures from 1980 to 2013 are selected and added together for each country. This produces a new column containing the cumulative immigration total across the specified period.

In [5]:
# Create a list of year columns from 1980 through 2013
years = [str(year) for year in range(1980, 2014)]

# Calculate the cumulative immigration figure for each country
df['Total'] = df[years].sum(axis=1)

# Display sample records and the updated dataset dimensions
print(df.head())
print("\nDimensions of the completed dataset:", df.shape)

               Continent           Region             DevName  1980  1981  \
Country                                                                     
Afghanistan         Asia    Southern Asia  Developing regions    16    39   
Albania           Europe  Southern Europe   Developed regions     1     0   
Algeria           Africa  Northern Africa  Developing regions    80    67   
American Samoa   Oceania        Polynesia  Developing regions     0     1   
Andorra           Europe  Southern Europe   Developed regions     0     0   

                1982  1983  1984  1985  1986  ...  2005  2006  2007  2008  \
Country                                       ...                           
Afghanistan       39    47    71   340   496  ...  3436  3009  2652  2111   
Albania            0     0     0     0     1  ...  1223   856   702   560   
Algeria           71    69    63    44    69  ...  3626  4807  3623  4005   
American Samoa     0     0     0     0     0  ...     0     1     0     0  

Step 6: Verify the Calculated Results

Selected country totals are displayed to check the calculations. The sum of the country-level totals is then compared with the original aggregate record to evaluate whether the calculations are consistent.

In [6]:
# Display the calculated totals for five selected countries
print(df.loc[
    ['Afghanistan', 'Albania', 'Algeria',
     'American Samoa', 'Andorra'],
    'Total'
])

# Calculate the combined total across all countries
print("\nCombined country totals:", df['Total'].sum())

# Calculate the aggregate figure from the original dataset
print(
    "Original aggregate total:",
    un_total_row.iloc[:, 9:].sum(axis=1).values[0]
)

Country
Afghanistan       58639
Albania           15699
Algeria           69439
American Samoa        6
Andorra              15
Name: Total, dtype: int64

Combined country totals: 6924212
Original aggregate total: 6924212


Step 7: Conclusion

The immigration dataset was prepared by reviewing its structure, identifying aggregate records, removing unnecessary columns, and organizing the remaining information by country. A new column was created to summarize immigration figures for each country between 1980 and 2013.

The final validation step examines selected country totals and compares the combined result with the aggregate figure from the original dataset. This process helps assess the consistency of the calculations and demonstrates how data cleaning, transformation, and validation contribute to reliable data analysis.